In [2]:
! pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.8 MB/s eta 0:00:00


In [1]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

# https://react-lm.github.io/ patrón de diseño ReAct

In [15]:
import os
from getpass import getpass
from groq import Groq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Ingresa GROQ_API_KEY: ")

# Seleccionamos un modelo de los que Groq dispone
MODEL = "qwen/qwen3.8-27b"

print("GROQ_API_KEY configurada:", bool(os.environ.get("GROQ_API_KEY")))

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Eres un asistente claro y didáctico."},
        {"role": "user", "content": "Explica en 3 líneas qué es un agente de IA."},
    ],
    temperature=0.2,
)

print(response.choices[0].message.content)

GROQ_API_KEY configurada: True
Un agente de IA es un sistema autónomo que percibe su entorno y toma decisiones para alcanzar objetivos específicos.
A diferencia de un chatbot pasivo, puede ejecutar acciones, usar herramientas y planificar pasos de forma independiente.
Su capacidad de actuar de manera proactiva lo convierte en un asistente funcional capaz de resolver tareas complejas.


In [16]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system}) # Esto es la identidad del agente

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message}) # Guarda los mensajes que envía el usuario
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result}) # Guarda los mensaje que envía el agente

        # Esto permite tener una mejor trazabilidad de los mensajes
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model=MODEL,
                        temperature=0, # Determinístico entre más cerca a cero.
                        messages=self.messages)
        return completion.choices[0].message.content


In [17]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [18]:
def calculate(what):
    return eval(what) # Permite evaluar expresiones que vienen como string

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [19]:
abot = Agent(prompt)
result = abot('How much does a toy poodel weigh?')
print(result)

Thought: I need to find the average weight of a Toy Poodle. I can use the average_dog_weight tool with the breed "Toy Poodle".
Action: average_dog_weight: Toy Poodle
PAUSE
